# Walkthrough: mnist_regularization_pytorch.ipynb

**Statistisches Lernen 2 — FH Kufstein Tirol**  
Arquivo de referência: `mnist_regularization_pytorch.ipynb`

Este walkthrough explica **passo a passo** cada célula do notebook original: o que foi feito, por que foi feito dessa forma, como cada ferramenta do PyTorch funciona, e como você deve proceder ao usá-las em seus próprios projetos.

---

> **Estrutura de cada seção:** O que o código faz → Como a ferramenta funciona → Por que foi feito assim → Como proceder / adaptar

---

## Visão geral do notebook original

O `mnist_regularization_pytorch.ipynb` treina um **MLP (Multi-Layer Perceptron)** no dataset MNIST e compara **8 estratégias de regularização** lado a lado, implementadas diretamente no loop de treinamento PyTorch.

| Seção | Conteúdo |
|-------|----------|
| 1 | Setup: imports, seed, device |
| 2 | Data Pipeline: MNIST, transforms, DataLoader |
| 3 | Model Architecture: ExplicitLinear, MLP, apply_constraints |
| 4 | Training Loop: loss, optimizer, Early Stopping |
| 5 | Experiments: 8 configurações de regularização |
| 6 | Visualization: filtros, heatmaps, histogramas de pesos |

**Regularizações cobertas:**

| Tipo | Método | Como é aplicado |
|------|--------|----------------|
| Soft | **L2 (Weight Decay)** | `weight_decay` no optimizer |
| Soft | **L1** | Termo extra na loss manualmente |
| Hard | **Max-Norm** | Projeção pós-optimizer |
| Hard | **Non-Negativity** | `clamp_(min=0)` pós-optimizer |
| Hard | **Weight Clipping** | `clamp_(-c, c)` pós-optimizer |
| Design | **Dropout** | `nn.Dropout` na forward pass |
| Output | **Label Smoothing** | `CrossEntropyLoss(label_smoothing=...)` |
| — | **Early Stopping** | Patience no loop de treino |

---

## Seção 1 — Setup: Imports, Seed e Device

### O que o código faz

```python
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
import torchvision
from torchvision import datasets, transforms

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
```

---

### `os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"`

Esta linha desabilita um erro que ocorre no Windows/Mac quando duas bibliotecas OpenMP são carregadas simultaneamente (PyTorch e NumPy às vezes conflitam). Não tem impacto no treinamento — é apenas uma correção de ambiente.

**Quando usar:** sempre que aparecer o erro `OMP: Error #15`. Em sistemas Linux, geralmente não é necessário.

---

### Estrutura dos imports do PyTorch

| Import | O que contém | Exemplos de uso |
|--------|-------------|----------------|
| `torch` | Tensores, operações básicas | `torch.tensor`, `torch.zeros`, `torch.no_grad` |
| `torch.nn` | Camadas prontas, losses | `nn.Linear`, `nn.CrossEntropyLoss`, `nn.Dropout` |
| `torch.nn.functional` | Funções sem estado | `F.relu`, `F.linear`, `F.softmax` |
| `torch.utils.data` | Dataset, DataLoader | `DataLoader`, `random_split`, `Subset` |
| `torchvision.datasets` | Datasets prontos | `datasets.MNIST`, `datasets.CIFAR10` |
| `torchvision.transforms` | Transformações de imagem | `transforms.ToTensor`, `transforms.Normalize` |

**`nn` vs `F`:** a diferença fundamental é que módulos em `nn` têm **estado** (parâmetros treináveis), enquanto funções em `F` são **stateless** (sem parâmetros). Ex: `nn.BatchNorm1d` mantém média e variância; `F.relu` não tem parâmetros.

---

### `set_seed` — por que reproducibilidade importa

Redes neurais usam **aleatoriedade** em vários pontos:
- Inicialização dos pesos (Kaiming, Xavier)
- Ordem dos batches no DataLoader (`shuffle=True`)
- Dropout (máscara aleatória)
- Divisão treino/validação (`random_split`)

Sem fixar a seed, **dois treinos com a mesma configuração produzem resultados diferentes**. Com `set_seed(42)`, qualquer pessoa que rodar o notebook verá os mesmos números.

É necessário fixar seeds em **quatro lugares** distintos porque cada biblioteca tem seu próprio gerador:
```python
random.seed(seed)           # biblioteca random do Python
np.random.seed(seed)        # NumPy
torch.manual_seed(seed)     # PyTorch CPU
torch.cuda.manual_seed_all(seed)  # PyTorch GPU (todos os dispositivos)
```

---

### `device` — CPU vs GPU

```python
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
```

O PyTorch suporta execução em **CPU** ou **GPU (CUDA)**. O código automaticamente usa GPU se disponível. Para usar o device, todo tensor e modelo devem ser movidos para ele:

```python
model = MLP().to(device)         # mover modelo
xb = xb.to(device)               # mover batch de dados
yb = yb.to(device)               # mover labels
```

**Nunca misture** tensores de CPU com tensores de GPU — gera erro imediatamente.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device em uso: {device}")
print(f"Versão do PyTorch: {torch.__version__}")

# Demonstração: o que é um Tensor PyTorch
print("\n--- O que é um Tensor PyTorch ---")
t = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(f"Tensor:\n{t}")
print(f"shape: {t.shape}  |  dtype: {t.dtype}  |  device: {t.device}")
print(f"Equivalente NumPy: {t.numpy()}")

# Demonstração de seed
print("\n--- Efeito da seed na reproducibilidade ---")
torch.manual_seed(0)
a = torch.randn(3)
torch.manual_seed(0)
b = torch.randn(3)
print(f"Com seed=0: {a}")
print(f"Com seed=0: {b}")
print(f"São iguais? {torch.allclose(a, b)}")
set_seed(42)  # restaurar seed do notebook

---

## Seção 2 — Data Pipeline: MNIST, Transforms e DataLoader

### O que o código faz

```python
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_full = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_ds    = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

val_size   = 1000
train_size = int(len(train_full)*0.1 - val_size)
indices    = torch.randperm(len(train_full), generator=...)[:(val_size + train_size)]
small_ds   = Subset(train_full, indices)
train_ds, val_ds = random_split(small_ds, [train_size, val_size], ...)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  ...)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False, ...)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, ...)
```

Resultado: **train=5000 / val=1000 / test=10000** amostras.

---

### O dataset MNIST

MNIST é o "hello world" do deep learning: 70.000 imagens de dígitos manuscritos (0–9), cada uma de tamanho **28×28 pixels** em escala de cinza. É um problema de **classificação multiclasse** com 10 classes.

- Train original: 60.000 imagens
- Test original: 10.000 imagens
- O notebook usa apenas **~10% do treino** (5.000) para tornar o experimento rápido e destacar o efeito da regularização com poucos dados.

---

### `transforms.Compose` — pipeline de pré-processamento

`transforms.Compose` encadeia transformações em sequência. Cada imagem passa por todas elas na ordem definida:

**`transforms.ToTensor()`**
- Converte imagem PIL (HxW, valores 0–255) para Tensor PyTorch (CxHxW, valores 0.0–1.0)
- Para MNIST: `(1, 28, 28)` — 1 canal (cinza), 28x28 pixels
- **Sempre necessário** para passar imagens para modelos PyTorch

**`transforms.Normalize((mean,), (std,))`**
- Aplica: `pixel_normalizado = (pixel - mean) / std`
- Para MNIST: `mean=0.1307`, `std=0.3081` (calculados sobre o dataset completo)
- **Por que normalizar?** Garante que os pixels tenham média ≈ 0 e desvio ≈ 1, o que:
  - Estabiliza o gradiente (evita saturação de ativações)
  - Acelera a convergência do optimizer
  - Torna o λ da regularização mais interpretável e comparável entre projetos

---

### `Subset` e `random_split` — dividir o dataset

| Ferramenta | O que faz |
|------------|----------|
| `torch.randperm(n)` | Permutação aleatória de 0 a n-1 (shuffle de índices) |
| `Subset(dataset, indices)` | Cria subconjunto de um dataset usando lista de índices |
| `random_split(dataset, [n1, n2])` | Divide dataset em dois de tamanhos n1 e n2 aleatoriamente |

**Por que usar `generator=torch.Generator().manual_seed(42)` nos dois lugares?**  
Para garantir que a divisão treino/validação seja **sempre a mesma** ao rodar o notebook novamente. Sem isso, cada execução colocaria amostras diferentes no conjunto de validação.

---

### `DataLoader` — o motor de batches

O `DataLoader` pega um dataset e entrega **batches** automaticamente durante o loop de treino.

| Parâmetro | O que controla | Valor no notebook | Quando mudar |
|-----------|---------------|-------------------|-------------|
| `batch_size` | Amostras por batch | 128 (treino), 256 (val/test) | Menor = mais ruído; Maior = mais estável mas mais RAM |
| `shuffle` | Embaralhar a cada epoch | `True` (treino), `False` (val/test) | Sempre True no treino; nunca em val/test |
| `num_workers` | Processos paralelos de leitura | 2 | 0 para debug; 4-8 para produção |
| `pin_memory` | Alocação em memória fixada para GPU | `True` | `True` se tiver GPU; `False` em CPU puro |

**Por que `shuffle=False` na validação?** Porque não importa a ordem em que os dados são avaliados — o resultado é o mesmo. E resultados reproduzíveis são mais fáceis de debugar.

In [ ]:
# Reproduzindo o data pipeline do notebook original
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_full = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_ds    = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

print(f"MNIST completo (treino): {len(train_full)} amostras")
print(f"MNIST completo (teste):  {len(test_ds)} amostras")

# Inspecionar uma amostra
img, label = train_full[0]
print(f"\nUma amostra:")
print(f"  Imagem shape: {img.shape}  (Canais x Altura x Largura)")
print(f"  Label: {label} (dígito '{label}')")
print(f"  Pixel min/max após normalize: {img.min():.3f} / {img.max():.3f}")

# Criar subset pequeno (igual ao notebook)
val_size   = 1000
train_size = int(len(train_full) * 0.1 - val_size)
indices    = torch.randperm(len(train_full),
                             generator=torch.Generator().manual_seed(42))[:val_size + train_size]
small_ds   = Subset(train_full, indices)
train_ds, val_ds = random_split(small_ds, [train_size, val_size],
                                 generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=0, pin_memory=False)

print(f"\nDivisão final:")
print(f"  Train: {len(train_ds)} amostras em {len(train_loader)} batches de 128")
print(f"  Val:   {len(val_ds)} amostras em {len(val_loader)} batches de 256")
print(f"  Test:  {len(test_ds)} amostras em {len(test_loader)} batches de 256")

# Visualizar algumas amostras do MNIST
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img_orig, lbl = train_full[i]
    ax.imshow(img_orig.squeeze(), cmap='gray')
    ax.set_title(str(lbl), fontsize=10)
    ax.axis('off')
plt.suptitle("Amostras do MNIST (após ToTensor + Normalize)", fontsize=12)
plt.tight_layout()
plt.show()

# Demonstrar o que um batch parece
xb, yb = next(iter(train_loader))
print(f"\nUm batch do train_loader:")
print(f"  xb.shape: {xb.shape}  (batch_size x C x H x W)")
print(f"  yb.shape: {yb.shape}  (batch_size,)")
print(f"  yb (labels): {yb[:16].tolist()}")

---

## Seção 3 — Model Architecture: ExplicitLinear, MLP e apply_constraints

### Por que criar `ExplicitLinear` em vez de usar `nn.Linear`?

O `nn.Linear` do PyTorch funciona perfeitamente para uso comum, mas **esconde os pesos internamente**. O notebook cria `ExplicitLinear` para que os pesos sejam **acessíveis e manipuláveis diretamente** — necessário para implementar as regularizações hardas (Max-Norm, Non-Negativity, Weight Clipping).

```python
model.fc1.weight   # acesso direto ao tensor de pesos
model.fc1.weight.data.clamp_(min=0)  # modificar diretamente
```

Com `nn.Linear` padrão, isso também é possível, mas o `ExplicitLinear` deixa o código mais explícito e educativo.

---

### Como funciona `nn.Module` — a base de todo modelo PyTorch

Todo modelo em PyTorch herda de `nn.Module`. Os métodos obrigatórios são:

```python
class MeuModelo(nn.Module):
    def __init__(self):
        super().__init__()          # OBRIGATÓRIO: inicializa o módulo
        self.camada = nn.Linear(...)  # registra subcamadas

    def forward(self, x):
        return self.camada(x)       # define o fluxo de dados
```

**O que `nn.Module` faz automaticamente:**
- Registra todos os `nn.Parameter` para o optimizer encontrá-los via `model.parameters()`
- Implementa `.to(device)` que move **tudo** (pesos, buffers) para o device
- Implementa `.train()` / `.eval()` que ativa/desativa Dropout e BatchNorm corretamente
- Implementa `.state_dict()` / `.load_state_dict()` para salvar/carregar pesos

---

### `nn.Parameter` — como os pesos se tornam treináveis

```python
self.weight = nn.Parameter(torch.empty(out_features, in_features))
```

`nn.Parameter` é um Tensor especial que:
1. É **automaticamente registrado** no módulo (aparece em `model.parameters()`)
2. Tem `requires_grad=True` por padrão (o gradiente é calculado durante backprop)
3. É incluído no `state_dict()` (salvo ao salvar o modelo)

Um tensor comum `torch.empty(...)` **não** seria treinado — o optimizer não o veria.

---

### Kaiming Initialization — por que não inicializar com zeros?

```python
nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
```

Inicializar com zeros faz com que todos os neurônios aprendam a mesma coisa (**symmetry breaking problem**). A inicialização de Kaiming distribui os pesos de forma a manter a variância dos ativações constante ao longo das camadas:

$$W \sim \mathcal{U}\!\left(-\sqrt{\frac{6}{(1+a^2) \cdot \text{fan\_in}}},\; \sqrt{\frac{6}{(1+a^2) \cdot \text{fan\_in}}}\right)$$

Onde `fan_in` é o número de entradas de cada neurônio e `a` é o slope negativo do LeakyReLU (para ReLU, `a=0`; o valor `math.sqrt(5)` usado aqui é o padrão do PyTorch para `nn.Linear`).

---

### Arquitetura do MLP

```
Input: 28×28 = 784 pixels
  ↓  nn.Flatten()
Vetor: (784,)
  ↓  ExplicitLinear(784 → 64)  [fc1]
  ↓  BatchNorm1d(64)            [opcional]
  ↓  F.relu()
  ↓  Dropout(p)                 [opcional]
  ↓  ExplicitLinear(64 → 10)   [fc2]
Logits: (10,)  — um por classe (dígito 0–9)
```

**`nn.Flatten()`:** converte o tensor `(B, 1, 28, 28)` para `(B, 784)`, onde B é o batch_size. Necessário porque camadas lineares esperam vetores, não imagens 2D.

**`F.relu(x)`:** aplica $\max(0, x)$ elemento a elemento — a função de ativação que introduz não-linearidade. Sem ela, empilhar camadas lineares seria equivalente a ter uma única camada linear.

**Logits vs Probabilidades:** a última camada não tem `softmax`. O PyTorch recomenda trabalhar com logits crus e usar `CrossEntropyLoss`, que aplica `log_softmax` internamente de forma numericamente estável.

---

### `apply_constraints` — Hard Regularization pós-optimizer

```python
@torch.no_grad()
def apply_constraints(model, cfg):
    for m in model.modules():
        if isinstance(m, ExplicitLinear):
            W = m.weight.data
            if cfg.get('nonneg', False):      W.clamp_(min=0.0)
            if cfg.get('max_norm', None):     # projeção na bola L2
            if cfg.get('clip_value', None):   W.clamp_(-c, c)
```

**Por que chamar APÓS o `optimizer.step()`?**  
O optimizer atualiza os pesos sem respeitar constraints. Depois do step, **projetamos** os pesos de volta ao conjunto viável. Esta é a abordagem de **Projected Gradient Descent**:

$$\theta_{t+1} = \text{Proj}_{\mathcal{C}}(\theta_t - \eta \nabla \mathcal{L}(\theta_t))$$

**`@torch.no_grad()`:** decorador que desativa o cálculo de gradientes dentro da função. Necessário porque estamos modificando `.data` diretamente — não queremos que essa operação entre no grafo computacional.

**`.data` vs o tensor direto:** acessar `m.weight.data` permite modificar o tensor sem afetar o grafo de autograd. Modificar `m.weight` diretamente causaria erros.

In [ ]:
# Reproduzindo o modelo do notebook original

class ExplicitLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        if bias:
            self.bias = nn.Parameter(torch.empty(out_features))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in = self.weight.size(1)
            bound = 1.0 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x):
        return F.linear(x, self.weight, self.bias)


class MLP(nn.Module):
    def __init__(self, hidden=64, dropout=0.0, use_bn=False):
        super().__init__()
        self.flatten  = nn.Flatten()
        self.fc1      = ExplicitLinear(28*28, hidden)
        self.bn1      = nn.BatchNorm1d(hidden) if use_bn else None
        self.dropout  = nn.Dropout(p=dropout)  if dropout > 0 else None
        self.fc2      = ExplicitLinear(hidden, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        if self.bn1 is not None:    x = self.bn1(x)
        x = F.relu(x)
        if self.dropout is not None: x = self.dropout(x)
        x = self.fc2(x)
        return x


@torch.no_grad()
def apply_constraints(model, cfg):
    for m in model.modules():
        if isinstance(m, ExplicitLinear):
            W = m.weight.data
            if cfg.get('nonneg', False):
                W.clamp_(min=0.0)
            max_norm = cfg.get('max_norm', None)
            if max_norm is not None:
                norms = W.norm(p=2, dim=1, keepdim=True)
                mask  = norms > max_norm
                if mask.any():
                    scale = torch.ones_like(norms)
                    scale[mask] = max_norm / norms[mask]
                    W.mul_(scale)
            clip_value = cfg.get('clip_value', None)
            if clip_value is not None:
                W.clamp_(-clip_value, clip_value)


# Inspecionar o modelo
set_seed(42)
modelo_demo = MLP(hidden=64, dropout=0.0, use_bn=False)

print("=== Arquitetura do MLP ===")
print(modelo_demo)
print()

total_params = sum(p.numel() for p in modelo_demo.parameters())
print(f"Total de parâmetros treináveis: {total_params:,}")
print(f"  fc1.weight shape: {modelo_demo.fc1.weight.shape}  → {28*28} × 64 = {28*28*64:,} pesos")
print(f"  fc1.bias   shape: {modelo_demo.fc1.bias.shape}")
print(f"  fc2.weight shape: {modelo_demo.fc2.weight.shape}  → 64 × 10 = 640 pesos")
print(f"  fc2.bias   shape: {modelo_demo.fc2.bias.shape}")

print()
print("--- Forward pass em um batch ---")
x_dummy = torch.randn(4, 1, 28, 28)   # 4 imagens falsas
with torch.no_grad():
    logits = modelo_demo(x_dummy)
print(f"Input shape:  {x_dummy.shape}")
print(f"Output shape: {logits.shape}  ← 4 amostras × 10 classes (logits)")
print(f"Logits[0]: {logits[0].detach().numpy().round(3)}")

print()
print("--- Diferença: nn.Parameter vs tensor comum ---")
print("Parâmetros registrados (visíveis pelo optimizer):")
for nome, param in modelo_demo.named_parameters():
    print(f"  {nome:25s}  shape={tuple(param.shape)}  requires_grad={param.requires_grad}")

---

## Seção 4 — Training Loop: Loss, Optimizer e Early Stopping

### O que o código faz — visão geral

A função `train_one_model(cfg, epochs, patience)` implementa o loop de treinamento completo:

```
Para cada epoch:
  Para cada batch:
    1. Forward pass → logits
    2. Calcular loss (CrossEntropy + L1 opcional)
    3. Backward pass → gradientes
    4. Gradient Clipping (opcional)
    5. optimizer.step() → atualizar pesos
    6. apply_constraints() → Hard Regularization
  Avaliar em validação
  Early Stopping check
```

---

### `CrossEntropyLoss` — a loss de classificação multiclasse

```python
criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
```

Para classificação com $K$ classes, a **Cross-Entropy Loss** é:

$$\mathcal{L}_{CE} = -\sum_{k=1}^{K} y_k \log \hat{p}_k$$

Onde $y_k$ é o label (0 ou 1, one-hot) e $\hat{p}_k = \text{softmax}(\text{logit}_k)$.

**No PyTorch:** `nn.CrossEntropyLoss` recebe **logits** (não softmax!) e labels como **inteiros** (não one-hot). Internamente combina `log_softmax + NLLLoss` de forma numericamente estável.

**Label Smoothing:** substitui o label duro $y_k \in \{0, 1\}$ por um label suavizado:

$$y_k^{\text{smooth}} = y_k(1 - \varepsilon) + \frac{\varepsilon}{K}$$

Com `label_smoothing=0.1` e $K=10$: a classe correta recebe $0.9 + 0.01 = 0.91$ e as erradas recebem $0.01$ cada. Isso evita que o modelo fique excessivamente confiante (**overconfident**) e melhora generalização.

---

### Adam Optimizer com `weight_decay` — L2 implícito

```python
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=wd)
```

**Adam** combina momentum adaptativo com aprendizado de taxa adaptativa por parâmetro. É o optimizer padrão para redes neurais na maioria dos casos.

**`weight_decay`** adiciona regularização L2 diretamente na atualização dos pesos:

$$\theta_{t+1} = \theta_t - \eta \left( \nabla \mathcal{L}(\theta_t) + \lambda \theta_t \right)$$

É matematicamente equivalente a adicionar $\lambda \|\theta\|^2$ na loss, mas implementado de forma mais eficiente.

---

### L1 Manual — por que não está no optimizer?

```python
if l1_lambda > 0:
    l1 = sum(m.weight.abs().sum() for m in model.modules()
             if isinstance(m, ExplicitLinear))
    loss = loss + l1_lambda * l1
```

O PyTorch **não tem L1 nativo no optimizer** (apenas L2 via `weight_decay`). Por isso, a penalidade L1 é calculada **como parte da loss** antes do `backward()`. Isso é possível porque PyTorch constrói um grafo computacional dinâmico — qualquer operação sobre tensores com `requires_grad=True` é automaticamente diferenciável.

---

### O loop de backpropagation — sequência obrigatória

```python
optimizer.zero_grad(set_to_none=True)   # 1. Zerar gradientes
logits = model(xb)                       # 2. Forward pass
loss   = criterion(logits, yb)           # 3. Calcular loss
loss.backward()                          # 4. Backward (calcular gradientes)
optimizer.step()                         # 5. Atualizar pesos
```

**Esta ordem é sempre a mesma. Nunca mudar.**

| Passo | Por que é necessário |
|-------|---------------------|
| `zero_grad()` | PyTorch **acumula** gradientes por padrão. Sem zerar, os gradientes do batch anterior somam com os do atual |
| `forward` | Constrói o grafo computacional dinamicamente |
| `backward()` | Percorre o grafo ao contrário (regra da cadeia) e calcula $\partial \mathcal{L} / \partial \theta$ para cada parâmetro |
| `step()` | Usa os gradientes calculados para atualizar $\theta$ |

**`set_to_none=True`:** em vez de preencher os gradientes com zeros, define como `None`. Mais eficiente em memória.

---

### Gradient Clipping

```python
if cfg.get('grad_clip', None) is not None:
    nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
```

Limita a norma total do gradiente a um valor máximo **antes** do `optimizer.step()`. Se $\|\nabla\| > c$, os gradientes são reescalados: $\nabla \leftarrow c \cdot \nabla / \|\nabla\|$.

**Quando usar:** quando ocorre **gradient explosion** (loss vai para NaN ou infinito durante treino). Comum em RNNs, mas pode ocorrer em MLPs com learning rate alto.

---

### Early Stopping — parar antes de overfittar

```python
if val_loss < best_val - 1e-4:
    best_val   = val_loss
    best_state = model.state_dict().copy()
    wait = 0
else:
    wait += 1
    if wait >= patience:
        break
```

**Lógica:** monitora a validation loss a cada epoch. Se ela não melhora por `patience` epochs seguidas, o treino para e o modelo é restaurado para o **melhor checkpoint** (`best_state`).

**`model.state_dict()`:** retorna um dicionário com todos os pesos e buffers do modelo como tensores. É o formato padrão para salvar/carregar modelos PyTorch.

**`1e-4` de tolerância:** melhoras menores que 0.0001 na loss de validação não contam como melhoria real — evita que ruído estatístico resete o contador desnecessariamente.

In [ ]:
# Reproduzindo as funções de treinamento do notebook original

def accuracy(logits, y):
    return (logits.argmax(1) == y).float().mean().item()

def evaluate(model, loader, criterion):
    model.eval()   # modo avaliação: desativa Dropout, usa stats fixas do BatchNorm
    total_loss, total_correct, total = 0.0, 0, 0
    with torch.no_grad():  # sem gradientes: mais rápido e menos memória
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss   = criterion(logits, yb)
            total_loss    += loss.item() * xb.size(0)   # .item() converte tensor→float
            total_correct += (logits.argmax(1) == yb).sum().item()
            total         += xb.size(0)
    return total_loss / total, total_correct / total


def train_one_model(cfg, epochs=8, patience=3, verbose=False):
    model = MLP(hidden=cfg.get('hidden', 64),
                dropout=cfg.get('dropout', 0.0),
                use_bn=cfg.get('use_bn', False)).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.get('label_smoothing', 0.0))
    optimizer = torch.optim.Adam(model.parameters(),
                                  lr=cfg.get('lr', 1e-3),
                                  weight_decay=cfg.get('l2_weight_decay', 0.0))
    l1_lambda = cfg.get('l1_lambda', 0.0)

    best_val, best_state, wait = float('inf'), None, 0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(1, epochs + 1):
        model.train()   # modo treino: ativa Dropout
        running_loss, n = 0.0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)   # 1. zerar gradientes
            logits = model(xb)                       # 2. forward
            loss   = criterion(logits, yb)           # 3. loss base

            if l1_lambda > 0:                        # 3b. L1 manual
                l1 = sum(m.weight.abs().sum() for m in model.modules()
                         if isinstance(m, ExplicitLinear))
                loss = loss + l1_lambda * l1

            loss.backward()                          # 4. backward

            if cfg.get('grad_clip', None) is not None:  # 4b. gradient clipping
                nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])

            optimizer.step()                         # 5. atualizar pesos
            apply_constraints(model, cfg)            # 6. hard constraints

            running_loss += loss.item() * xb.size(0)
            n += xb.size(0)

        train_loss = running_loss / n
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if verbose:
            print(f"  Epoch {epoch:2d}: train={train_loss:.4f}  val={val_loss:.4f}  acc={val_acc:.4f}")

        if cfg.get('use_early_stopping', True):
            if val_loss < best_val - 1e-4:
                best_val   = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                wait       = 0
            else:
                wait += 1
                if wait >= patience:
                    if verbose: print(f"  Early stopping na epoch {epoch}")
                    break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_acc = evaluate(model, test_loader, criterion)
    return model, history, {'val_loss': best_val, 'test_acc': test_acc}


# Demonstrações isoladas de conceitos
print("=== Demonstração: CrossEntropyLoss ===")
logits_demo = torch.tensor([[2.0, 1.0, 0.5],   # amostra 0: mais confiante na classe 0
                              [0.1, 3.0, 0.2]])  # amostra 1: mais confiante na classe 1
labels_demo = torch.tensor([0, 1])              # labels corretos
loss_fn = nn.CrossEntropyLoss()
loss_val = loss_fn(logits_demo, labels_demo)
print(f"  Logits: {logits_demo.tolist()}")
print(f"  Labels: {labels_demo.tolist()}")
print(f"  CrossEntropy Loss: {loss_val.item():.4f}  (baixo = previsões corretas e confiantes)")

print()
print("=== Demonstração: Label Smoothing ===")
loss_smooth = nn.CrossEntropyLoss(label_smoothing=0.1)
print(f"  Sem smoothing:  {loss_fn(logits_demo, labels_demo).item():.4f}")
print(f"  Com smoothing:  {loss_smooth(logits_demo, labels_demo).item():.4f}")
print("  (Label smoothing aumenta a loss — o modelo não pode ser 100% confiante)")

print()
print("=== Demonstração: model.train() vs model.eval() ===")
mlp_drop = MLP(dropout=0.5)
x_test = torch.ones(1, 1, 28, 28)
mlp_drop.train()
out_train = mlp_drop(x_test).detach()
mlp_drop.eval()
out_eval  = mlp_drop(x_test).detach()
print(f"  Saída em .train() (dropout ativo): {out_train[0].numpy().round(3)}")
print(f"  Saída em .eval()  (dropout off):   {out_eval[0].numpy().round(3)}")
print("  (Valores diferentes — dropout zera neurônios aleatoriamente durante treino)")

---

## Seção 5 — Experiments: As 8 Configurações de Regularização

### O que o código faz

```python
experiments = [
    {'name': 'baseline',      ...  'l2_weight_decay': 0.0, 'l1_lambda': 0.0},
    {'name': 'l2',            ...  'l2_weight_decay': 1e-4},
    {'name': 'l1',            ...  'l1_lambda': 1e-6},
    {'name': 'maxnorm',       ...  'max_norm': 3.0},
    {'name': 'nonneg',        ...  'nonneg': True},
    {'name': 'clipW',         ...  'clip_value': 0.5},
    {'name': 'dropout',       ...  'dropout': 0.5},
    {'name': 'label_smooth',  ...  'label_smoothing': 0.1},
]
```

Cada configuração treina um modelo do zero com a mesma arquitetura e compara os resultados.

---

### Explicação detalhada de cada experimento

**Baseline** — sem regularização  
O ponto de referência. Treina sem nenhuma restrição adicional. Com apenas 5000 amostras, espera-se algum overfitting.

**L2 (`weight_decay=1e-4`)** — Soft Regularization  
Penaliza pesos grandes com $\lambda \|\theta\|^2$. Implementado diretamente no Adam via `weight_decay`. Os pesos tendem a ser pequenos mas não exatamente zero. Valor típico: `1e-6 … 1e-2`.

**L1 (`l1_lambda=1e-6`)** — Soft Regularization  
Penaliza a soma dos valores absolutos dos pesos $\lambda \|\theta\|_1$. Induz **sparsity**: força pesos irrelevantes para exatamente zero. Valor muito pequeno (`1e-6`) porque a soma de todos os pesos de uma rede é grande — um λ grande destruiria o modelo.

**Max-Norm (`max_norm=3.0`)** — Hard Regularization  
Limita a **norma L2 de cada linha** da matriz de pesos (= cada neurônio) a no máximo 3.0. Se um neurônio crescer demais, seus pesos são reescalados proporcionalemente. Estabiliza o treino sem impedir que os pesos aprendam direções importantes.

**Non-Negativity (`nonneg=True`)** — Hard Regularization  
Força todos os pesos a serem $\geq 0$ via `clamp_(min=0)`. Raramente usada em geral, mas útil quando o problema tem estrutura que requer pesos não-negativos (ex: decomposição de matrizes NMF, modelos de atenção positiva). Resultado: perda de accuracy porque nega metade do espaço de soluções.

**Weight Clipping (`clip_value=0.5`)** — Hard Regularization  
Limita cada peso individualmente ao intervalo $[-0.5, 0.5]$ via `clamp_(-c, c)`. Diferente do Max-Norm, age elemento a elemento. Conhecido do treinamento de **Wasserstein GANs** (onde é aplicado para satisfazer a condição de Lipschitz).

**Dropout (`dropout=0.5`)** — Regularization by Design  
Zerando 50% dos neurônios aleatoriamente a cada forward pass de treino. O modelo é forçado a aprender **representações redundantes** — como um ensemble implícito de $2^{64}$ sub-redes. Na avaliação, todos os neurônios estão ativos (os pesos são escalados por $1-p$ para compensar).

**Label Smoothing (`label_smoothing=0.1`)** — Output Regularization  
Suaviza os labels de treino: em vez de $y = [0,0,1,0,...]$ (hard), usa $y = [0.01, 0.01, 0.91, 0.01, ...]$. O modelo não pode maximizar confiança ilimitadamente. Resultado: **test_acc mais alto** que o baseline — comum em problemas de classificação.

---

### Resultados do notebook (referência)

```
baseline     | val_loss: 0.3049 | test_acc: 0.9237
l2           | val_loss: 0.3225 | test_acc: 0.9197
l1           | val_loss: 0.3236 | test_acc: 0.9190
maxnorm      | val_loss: 0.3093 | test_acc: 0.9212
nonneg       | val_loss: 0.4162 | test_acc: 0.8977  ← restrição muito severa
clipW        | val_loss: 0.3261 | test_acc: 0.9190
dropout      | val_loss: 0.3343 | test_acc: 0.9208
label_smooth | val_loss: 0.7700 | test_acc: 0.9336  ← val_loss alta (esperado!)
```

**Por que o val_loss do label_smooth é tão alto?** A CrossEntropyLoss com `label_smoothing=0.1` produz valores de loss maiores por design (os targets nunca são 1.0). Comparar `val_loss` entre modelos com e sem smoothing é uma comparação injusta — use `test_acc` para a comparação correta.

In [ ]:
# Rodar os experimentos (reproduzindo o notebook original)
set_seed(42)

experiments = [
    {'name': 'baseline',     'hidden': 64, 'dropout': 0.0, 'use_bn': False,
     'l2_weight_decay': 0.0, 'l1_lambda': 0.0, 'use_early_stopping': True},
    {'name': 'l2',           'l2_weight_decay': 1e-4},
    {'name': 'l1',           'l1_lambda': 1e-6},
    {'name': 'maxnorm',      'max_norm': 3.0},
    {'name': 'nonneg',       'nonneg': True},
    {'name': 'clipW',        'clip_value': 0.5},
    {'name': 'dropout',      'dropout': 0.5},
    {'name': 'label_smooth', 'label_smoothing': 0.1},
]

results, trained = [], []
for cfg in experiments:
    base = {'name': 'exp', 'hidden': 64, 'dropout': 0.0, 'use_bn': False,
            'l2_weight_decay': 0.0, 'l1_lambda': 0.0, 'nonneg': False,
            'max_norm': None, 'clip_value': None, 'label_smoothing': 0.0,
            'use_early_stopping': True, 'lr': 1e-3}
    base.update(cfg)
    print(f"Treinando: {base['name']}...")
    model, history, stats = train_one_model(base, epochs=8, patience=3, verbose=False)
    results.append((base['name'], history, stats))
    trained.append((base['name'], model, history, stats))

print("\n=== Resultados ===")
print(f"{'Método':12s} | {'Val Loss':>10s} | {'Test Acc':>10s}")
print("-" * 40)
for name, history, stats in results:
    print(f"{name:12s} | {stats['val_loss']:10.4f} | {stats['test_acc']:10.4f}")

---

## Seção 6 — Visualization: Filtros, Heatmaps e Histogramas de Pesos

### O que o código faz

O notebook original define 4 funções de visualização:

1. **`plot_fc1_filters`** — visualiza os pesos da primeira camada como imagens 28×28
2. **`plot_fc2_heatmap`** — visualiza a matriz de pesos da segunda camada como heatmap
3. **`plot_weight_hist`** — histograma da distribuição de todos os pesos de uma camada
4. **`weight_stats`** — métricas numéricas dos pesos (L1, L2, max, sparsity)

---

### `plot_fc1_filters` — ver o que o modelo aprendeu

```python
W = model.fc1.weight.detach().cpu()  # shape: (64, 784)
img = W[i].reshape(28, 28)           # cada neurônio como imagem
```

Cada neurônio da primeira camada tem **784 pesos** — um por pixel. Visualizá-los como imagens 28×28 revela o que cada neurônio está procurando na imagem:
- Pesos **positivos (vermelho):** pixels que ativam o neurônio
- Pesos **negativos (azul):** pixels que inibem o neurônio
- Pesos próximos de zero: pixels ignorados

**`detach()`:** remove o tensor do grafo computacional. Necessário antes de converter para NumPy ou fazer operações fora do contexto de treinamento.

**`.cpu()`:** move o tensor para CPU antes de plotar (Matplotlib não suporta tensores CUDA).

---

### `plot_fc2_heatmap` — conexões entre hidden e classes

```python
W = model.fc2.weight.detach().cpu()  # shape: (10, 64)
plt.imshow(W, aspect='auto', cmap='coolwarm')
```

A segunda camada conecta os 64 neurônios hidden às 10 classes. Cada linha é uma classe (0–9); cada coluna é um neurônio hidden. Cores intensas indicam neurônios muito importantes para aquela classe.

**Efeito da regularização:** modelos com L1 ou Weight Clipping terão heatmaps mais esparsos/uniformes; Dropout tende a distribuir a importância por mais neurônios.

---

### `weight_stats` — métricas numéricas para comparação

```python
{
    'L1_norm':  W.abs().sum(),
    'L2_norm':  W.pow(2).sum().sqrt(),
    'max_abs':  W.abs().max(),
    'sparsity_frac(|w|<1e-3)': (W.abs() < 1e-3).float().mean()
}
```

| Métrica | O que mede | Efeito esperado por regularização |
|---------|-----------|-----------------------------------|
| **L1 norm** | Soma dos abs dos pesos | L1 regularização → menor L1 norm |
| **L2 norm** | Comprimento do vetor de pesos | L2 regularização → menor L2 norm |
| **max_abs** | Maior peso em valor absoluto | Max-Norm → limitado a `max_norm` |
| **sparsity_frac** | Fração de pesos quase-zero | L1 → alta sparsity; L2 → baixa |

---

### Como usar o `colormap` `coolwarm`

`coolwarm` é ideal para dados com **ponto central em zero** (divergindo positivo/negativo):
- **Azul** → valores negativos
- **Branco** → zero
- **Vermelho** → valores positivos

Sempre combine com `vmin=-vmax, vmax=vmax` para centralizar o colormap em zero. Sem isso, o branco pode não corresponder a zero.

In [ ]:
# Reproduzindo as funções de visualização do notebook original

def plot_fc1_filters(model, name, n=16, cmap='coolwarm'):
    W = model.fc1.weight.detach().cpu()
    vmax = float(W.abs().max())
    k = int(math.sqrt(n))
    fig, axes = plt.subplots(k, k, figsize=(2*k, 2*k))
    for i, ax in enumerate(axes.flat[:n]):
        img = W[i].reshape(28, 28)
        im  = ax.imshow(img, cmap=cmap, vmin=-vmax, vmax=vmax)
        ax.set_title(f'N{i}', fontsize=7)
        ax.axis('off')
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6)
    plt.suptitle(f'fc1: pesos como filtros 28×28  ({name})', fontsize=11)
    plt.tight_layout()
    plt.show()

def plot_fc2_heatmap(model, name, cmap='coolwarm'):
    W = model.fc2.weight.detach().cpu()   # (10, 64)
    vmax = float(W.abs().max())
    plt.figure(figsize=(10, 3))
    plt.imshow(W, aspect='auto', cmap=cmap, vmin=-vmax, vmax=vmax)
    plt.colorbar(shrink=0.7)
    plt.yticks(range(10), [str(i) for i in range(10)])
    plt.xlabel('Hidden Neuron index'); plt.ylabel('Classe (dígito)')
    plt.title(f'fc2: matriz de pesos (10 classes × 64 neurônios)  ({name})')
    plt.tight_layout(); plt.show()

def plot_weight_hist(model, name, layer='fc1'):
    W = getattr(model, layer).weight.detach().cpu().view(-1)
    plt.figure(figsize=(6, 3))
    plt.hist(W.numpy(), bins=60, color='steelblue', alpha=0.85, edgecolor='white')
    plt.axvline(0, color='red', lw=1.5, ls='--')
    plt.title(f'Histograma de pesos {layer}  ({name})')
    plt.xlabel('Valor do peso'); plt.ylabel('Frequência')
    plt.tight_layout(); plt.show()

def weight_stats(model):
    stats = {}
    for lname in ['fc1', 'fc2']:
        W = getattr(model, lname).weight.detach().cpu()
        stats[lname] = {
            'L1_norm':    float(W.abs().sum()),
            'L2_norm':    float(W.pow(2).sum().sqrt()),
            'max_abs':    float(W.abs().max()),
            'sparsity':   float((W.abs() < 1e-3).float().mean()),
        }
    return stats


# Comparar baseline vs L1 vs nonneg visualmente
nomes_demo = ['baseline', 'l1', 'nonneg', 'label_smooth']
modelos_demo = {name: model for name, model, _, _ in trained if name in nomes_demo}

for nome in nomes_demo:
    if nome not in modelos_demo:
        continue
    mdl = modelos_demo[nome]
    print(f"\n{'='*50}")
    print(f"Modelo: {nome}")
    plot_fc1_filters(mdl, nome, n=16)
    plot_fc2_heatmap(mdl, nome)
    plot_weight_hist(mdl, nome, layer='fc1')
    st = weight_stats(mdl)
    print(f"  fc1: L1={st['fc1']['L1_norm']:.1f}  L2={st['fc1']['L2_norm']:.2f}  "
          f"max={st['fc1']['max_abs']:.3f}  sparsity={st['fc1']['sparsity']:.3f}")

In [ ]:
# Comparação visual das learning curves de todos os experimentos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cores = plt.cm.tab10(np.linspace(0, 1, len(results)))

ax = axes[0]
for (name, history, _), cor in zip(results, cores):
    ax.plot(history['val_loss'], label=name, color=cor, lw=2)
ax.set_title('Val Loss por epoch')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val Loss')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
for (name, history, _), cor in zip(results, cores):
    ax.plot(history['val_acc'], label=name, color=cor, lw=2)
ax.set_title('Val Accuracy por epoch')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val Accuracy')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Comparação de todas as regularizações', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Bar chart de test accuracy
nomes  = [r[0] for r in results]
accs   = [r[2]['test_acc'] for r in results]
baseline_acc = dict(zip(nomes, accs))['baseline']

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(nomes, accs, color=cores, alpha=0.8, edgecolor='black')
ax.axhline(baseline_acc, color='black', ls='--', lw=1.5, label=f'Baseline={baseline_acc:.4f}')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.001, f'{acc:.4f}',
            ha='center', va='bottom', fontsize=8)
ax.set_title('Test Accuracy por método de regularização')
ax.set_ylabel('Test Accuracy')
ax.set_ylim(min(accs) - 0.01, max(accs) + 0.01)
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Comparar weight stats entre todos os modelos
print("\n=== Weight Stats Comparadas (fc1) ===")
print(f"{'Método':12s} | {'L1 norm':>10s} | {'L2 norm':>10s} | {'max|w|':>8s} | {'sparsity':>10s}")
print('-' * 60)
for name, model, _, _ in trained:
    st = weight_stats(model)
    s = st['fc1']
    print(f"{name:12s} | {s['L1_norm']:10.1f} | {s['L2_norm']:10.3f} | "
          f"{s['max_abs']:8.4f} | {s['sparsity']:10.4f}")

---

## Guia de ferramentas PyTorch: referência rápida

### Estrutura de um projeto PyTorch

```
1. Dataset + DataLoader
2. Modelo (nn.Module)
3. Loss Function
4. Optimizer
5. Training Loop
6. Evaluation
```

### Ferramentas usadas no notebook

| Ferramenta | Sintaxe | O que faz |
|------------|---------|----------|
| Transformações | `transforms.Compose([...])` | Pipeline sequencial de pré-processamento |
| Dataset MNIST | `datasets.MNIST(root, train, download, transform)` | Baixa e carrega MNIST |
| Subconjunto | `Subset(dataset, indices)` | Seleciona subconjunto por índices |
| Divisão | `random_split(dataset, [n1, n2])` | Divide aleatoriamente em partes |
| DataLoader | `DataLoader(ds, batch_size, shuffle)` | Entrega batches durante o loop |
| Parâmetro treinável | `nn.Parameter(tensor)` | Tensor que o optimizer atualiza |
| Inicialização | `nn.init.kaiming_uniform_(W, a=...)` | Inicialização de Kaiming |
| Flatten | `nn.Flatten()` | Achata dimensões: (B,C,H,W)→(B, C*H*W) |
| ReLU | `F.relu(x)` | Ativação: max(0, x) |
| Dropout | `nn.Dropout(p=0.5)` | Zera p% dos neurônios durante treino |
| BatchNorm | `nn.BatchNorm1d(features)` | Normaliza ativações por batch |
| Loss | `nn.CrossEntropyLoss(label_smoothing=...)` | Loss de classificação multiclasse |
| Optimizer | `torch.optim.Adam(params, lr, weight_decay)` | Otimizador Adam com L2 opcional |
| Zerar grads | `optimizer.zero_grad(set_to_none=True)` | Limpa gradientes acumulados |
| Backprop | `loss.backward()` | Calcula gradientes via autograd |
| Atualizar pesos | `optimizer.step()` | Aplica atualização dos parâmetros |
| Grad clipping | `nn.utils.clip_grad_norm_(params, max)` | Limita norma dos gradientes |
| Sem gradientes | `with torch.no_grad():` | Desativa autograd (avaliação) |
| Sem gradientes (fn) | `@torch.no_grad()` | Decorador para funções de avaliação |
| Salvar pesos | `model.state_dict()` | Dicionário com todos os pesos |
| Carregar pesos | `model.load_state_dict(state)` | Restaura pesos salvos |
| Mover para device | `.to(device)` | Move modelo/tensor para CPU ou GPU |
| Detach | `.detach()` | Remove tensor do grafo computacional |
| Para numpy | `.cpu().numpy()` | Converte tensor para array NumPy |

---

## Checklist: como adaptar para um novo dataset

**1. Substituir o Dataset**
```python
# Para imagens customizadas:
from torchvision.datasets import ImageFolder
ds = ImageFolder(root='caminho/para/pasta', transform=transform)

# Para dados tabulares:
from torch.utils.data import TensorDataset
ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                   torch.tensor(y, dtype=torch.long))
```

**2. Adaptar o modelo (input/output)**
```python
# Mudar 28*28 para o tamanho do seu input
# Mudar 10 para o número de classes
self.fc1 = ExplicitLinear(seu_input_size, hidden)
self.fc2 = ExplicitLinear(hidden, seu_n_classes)
```

**3. Escolher a regularização (regra prática)**
```
Poucos dados, muitas features  → L2 (weight_decay=1e-4) + Early Stopping
Quer feature selection         → L1 (l1_lambda=1e-5)
Treino instável (loss=NaN)     → Gradient Clipping (grad_clip=1.0) + Max-Norm
Classificação, muitas classes  → Label Smoothing (0.05–0.2)
Modelo grande, dados médios    → Dropout (0.3–0.5) nas camadas finais
```

**4. Diagnosticar pela val_loss**
```
val_loss >> train_loss e grande gap → Overfitting → aumentar regularização
val_loss ≈ train_loss mas ambos altos → Underfitting → reduzir regularização ou aumentar modelo
val_loss aumenta após certa epoch → Early Stopping vai salvar o melhor modelo
```

---

## Exercícios para consolidar

1. **Arquitetura:** adicione uma terceira camada hidden ao `MLP` e veja o efeito no test_acc.
2. **Elastic Net:** combine L1 e L2 no mesmo experimento — `l1_lambda=1e-6` + `l2_weight_decay=1e-4`.
3. **Tuning:** use o `Coarse-to-Fine` do notebook `visao_probabilistica.ipynb` para encontrar o melhor `weight_decay` para Ridge.
4. **BatchNorm:** adicione `use_bn=True` ao baseline. O BatchNorm funciona como regularizador? Compare val_loss.
5. **Salvar/Carregar:** após treinar o baseline, salve com `torch.save(model.state_dict(), 'baseline.pt')` e carregue em uma nova instância de `MLP`.
6. **GPU:** se tiver acesso a GPU (Google Colab), mova tudo para `device='cuda'` e compare o tempo de treino.
7. **Weight Histograms:** compare os histogramas de `fc1` do baseline vs `nonneg` — como a distribuição muda?